# Lab4-Assignment about Named Entity Recognition and Classification

This notebook describes the assignment of Lab 4 of the text mining course. We assume you have succesfully completed Lab1, Lab2 and Lab3 as welll. Especially Lab2 is important for completing this assignment.

**Learning goals**
* going from linguistic input format to representing it in a feature space
* working with pretrained word embeddings
* train a supervised classifier (SVM)
* evaluate a supervised classifier (SVM)
* learn how to interpret the system output and the evaluation results
* be able to propose future improvements based on the observed results


## Credits
This notebook was originally created by [Marten Postma](https://martenpostma.github.io) and [Filip Ilievski](http://ilievski.nl) and adapted by Piek vossen

## [Points: 18] Exercise 1 (NERC): Training and evaluating an SVM using CoNLL-2003

**[4 point] a) Load the CoNLL-2003 training data using the *ConllCorpusReader* and create for both *train.txt* and *test.txt*:**

    [2 points]  -a list of dictionaries representing the features for each training instances, e..g,
    ```
    [
    {'words': 'EU', 'pos': 'NNP'}, 
    {'words': 'rejects', 'pos': 'VBZ'},
    ...
    ]
    ```

    [2 points] -the NERC labels associated with each training instance, e.g.,
    dictionaries, e.g.,
    ```
    [
    'B-ORG', 
    'O',
    ....
    ]
    ```

In [4]:
from nltk.corpus.reader import ConllCorpusReader
import numpy as np
import gensim

word_embedding_model = gensim.models.KeyedVectors.load_word2vec_format(r"../Lab_2/Google_News_files/GoogleNews-vectors-negative300.bin.gz", binary = True)

### Adapt the path to point to the CONLL2003 folder on your local machine
train = ConllCorpusReader('CONLL2003', 'train.txt', ['words', 'pos', 'ignore', 'chunk'])
training_features = []
training_gold_labels = []


for token, pos, ne_label in train.iob_words():
    if token != '' and token != 'DOCSTART':
        a_dict = {
            "words": token,
            "pos": pos,
            "Is_upper": token.isupper(),
            "Is_lower": token.islower(),
            "Is_title": token.istitle(),
            "Is_number": token.isdigit(),
        }
                    
        
        training_features.append(a_dict)
        training_gold_labels.append(ne_label)


print(training_features[0], training_features[1], training_features[2])
print(training_gold_labels[0], training_gold_labels[1], training_gold_labels[2])

{'words': 'EU', 'pos': 'NNP', 'Is_upper': True, 'Is_lower': False, 'Is_title': False, 'Is_number': False} {'words': 'rejects', 'pos': 'VBZ', 'Is_upper': False, 'Is_lower': True, 'Is_title': False, 'Is_number': False} {'words': 'German', 'pos': 'JJ', 'Is_upper': False, 'Is_lower': False, 'Is_title': True, 'Is_number': False}
B-ORG O B-MISC


In [5]:
### Adapt the path to point to the CONLL2003 folder on your local machine
train = ConllCorpusReader('CONLL2003', 'test.txt', ['words', 'pos', 'ignore', 'chunk'])

test_features = []
test_gold_labels = []

for token, pos, ne_label in train.iob_words():
    if token != '' and token != 'DOCSTART':
        a_dict = {
            "words": token,
            "pos": pos,
            "Is_upper": token.isupper(),
            "Is_lower": token.islower(),
            "Is_title": token.istitle(),
            "Is_number": token.isdigit(),
        }
        
        
        
        test_features.append(a_dict)
        test_gold_labels.append(ne_label)

print(test_features[0], training_features[1], training_features[2])
print(test_gold_labels[0], test_gold_labels[1], test_gold_labels[2])

{'words': 'SOCCER', 'pos': 'NN', 'Is_upper': True, 'Is_lower': False, 'Is_title': False, 'Is_number': False} {'words': 'rejects', 'pos': 'VBZ', 'Is_upper': False, 'Is_lower': True, 'Is_title': False, 'Is_number': False} {'words': 'German', 'pos': 'JJ', 'Is_upper': False, 'Is_lower': False, 'Is_title': True, 'Is_number': False}
O O B-LOC


In [6]:
print(training_features[0])

{'words': 'EU', 'pos': 'NNP', 'Is_upper': True, 'Is_lower': False, 'Is_title': False, 'Is_number': False}


In [7]:
print(training_features[1:3])
print(training_gold_labels[0:10])

[{'words': 'rejects', 'pos': 'VBZ', 'Is_upper': False, 'Is_lower': True, 'Is_title': False, 'Is_number': False}, {'words': 'German', 'pos': 'JJ', 'Is_upper': False, 'Is_lower': False, 'Is_title': True, 'Is_number': False}]
['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O', 'B-PER']


In [8]:
print(type(training_features), type(training_gold_labels))

<class 'list'> <class 'list'>


**[2 points] b) provide descriptive statistics about the training and test data:**
* How many instances are in train and test?
* Provide a frequency distribution of the NERC labels, i.e., how many times does each NERC label occur?
* Discuss to what extent the training and test data is balanced (equal amount of instances for each NERC label) and to what extent the training and test data differ?

Tip: you can use the following `Counter` functionality to generate frequency list of a list:

In [9]:
from collections import Counter


print(f'Length of training list: {len(training_features)}\n')
print(f'Length of test list: {len(test_features)}\n')
print(f'Frequency distribution of NERC labels in training list: {Counter(training_gold_labels)}\n')
print(f'Frequency distribution of NERC labels in training list: {Counter(test_gold_labels)}')

Length of training list: 203621

Length of test list: 46435

Frequency distribution of NERC labels in training list: Counter({'O': 169578, 'B-LOC': 7140, 'B-PER': 6600, 'B-ORG': 6321, 'I-PER': 4528, 'I-ORG': 3704, 'B-MISC': 3438, 'I-LOC': 1157, 'I-MISC': 1155})

Frequency distribution of NERC labels in training list: Counter({'O': 38323, 'B-LOC': 1668, 'B-ORG': 1661, 'B-PER': 1617, 'I-PER': 1156, 'I-ORG': 835, 'B-MISC': 702, 'I-LOC': 257, 'I-MISC': 216})


In [10]:
my_list=[1,2,1,3,2,5]
Counter(my_list)


Counter({1: 2, 2: 2, 3: 1, 5: 1})

## Analysis of data statistics



**[2 points] c) Concatenate the train and test features (the list of dictionaries) into one list. Load it using the *DictVectorizer*. Afterwards, split it back to training and test.**

Tip: You’ve concatenated train and test into one list and then you’ve applied the DictVectorizer.
The order of the rows is maintained. You can hence use an index (number of training instances) to split the_array back into train and test. Do NOT use: `
from sklearn.model_selection import train_test_split` here.


In [11]:
from sklearn.feature_extraction import DictVectorizer

In [12]:
vec = DictVectorizer(sparse=False)

# Concatenate training and test features
all_features = training_features + test_features

# Apply vectorization
all_feature_matrix = vec.fit_transform(all_features)

# Split the vectorized features back into training and testing sets based on the number of training instances
num_train = len(training_features)
train_feature_matrix = all_feature_matrix[:num_train]
test_feature_matrix = all_feature_matrix[num_train:]

training_gold_labels = np.array(training_gold_labels)
test_gold_labels = np.array(test_gold_labels)

print(f"Train matrix: {train_feature_matrix}")
print(f"Test matrix: {test_feature_matrix}")
print()
print(f"Shape of training feature matrix: {train_feature_matrix.shape}, {type(train_feature_matrix)}")
print(f"Shape of test feature matrix: {test_feature_matrix.shape}, {type(test_feature_matrix)}")
print(f'Shape of training labels: {training_gold_labels.shape}')
print(f'Shape of testing labels: {test_gold_labels.shape}')

Train matrix: [[0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 ...
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]]
Test matrix: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]

Shape of training feature matrix: (203621, 27365), <class 'numpy.ndarray'>
Shape of test feature matrix: (46435, 27365), <class 'numpy.ndarray'>
Shape of training labels: (203621,)
Shape of testing labels: (46435,)


**[4 points] d) Train the SVM using the train features and labels and evaluate on the test data. Provide a classification report (sklearn.metrics.classification_report).**
The train (*lin_clf.fit*) might take a while. On my computer, it took 1min 53s, which is acceptable. Training models normally takes much longer. If it takes more than 5 minutes, you can use a subset for training. Describe the results:
* Which NERC labels does the classifier perform well on? Why do you think this is the case?
* Which NERC labels does the classifier perform poorly on? Why do you think this is the case?

In [13]:
# Check the shapes of the feature matrices and labels
print(f"Shape of training labels: {training_gold_labels.shape}")
print(f"Shape of testing labels: {test_gold_labels.shape}")
print(f"Shape of training feature matrix: {train_feature_matrix.shape}")
print(f"Shape of testing feature matrix: {test_feature_matrix.shape}")


Shape of training labels: (203621,)
Shape of testing labels: (46435,)
Shape of training feature matrix: (203621, 27365)
Shape of testing feature matrix: (46435, 27365)


In [14]:
from sklearn import svm
from sklearn.metrics import classification_report

In [15]:
lin_clf = svm.LinearSVC(max_iter=50000)

In [16]:
lin_clf.fit(train_feature_matrix, training_gold_labels)

LinearSVC(max_iter=50000)

In [17]:
test_predictions = lin_clf.predict(test_feature_matrix)

In [18]:
print(f"Length of test_gold_labels: {len(test_gold_labels)}")
print(f"Length of test_predictions: {len(test_predictions)}")
print(f"Shape of test_feature_matrix: {test_feature_matrix.shape}")
print(f"Shape of test_gold_labels: {test_gold_labels.shape}")
print(f"Length of training labels: {len(training_gold_labels)}")
print(f"Shape of training feature matrix: {train_feature_matrix.shape}")
print(f"Original training size: {len(training_features)}")
print(f"Vectorized training matrix size: {train_feature_matrix.shape[0]}")
train_feature_matrix.shape[0] == len(training_gold_labels)
# All data matched up and is ready to be tested

Length of test_gold_labels: 46435
Length of test_predictions: 46435
Shape of test_feature_matrix: (46435, 27365)
Shape of test_gold_labels: (46435,)
Length of training labels: 203621
Shape of training feature matrix: (203621, 27365)
Original training size: 203621
Vectorized training matrix size: 203621


True

In [19]:
print(classification_report(test_gold_labels, test_predictions))

              precision    recall  f1-score   support

       B-LOC       0.81      0.78      0.79      1668
      B-MISC       0.72      0.68      0.70       702
       B-ORG       0.80      0.53      0.63      1661
       B-PER       0.81      0.45      0.58      1617
       I-LOC       0.62      0.53      0.57       257
      I-MISC       0.57      0.59      0.58       216
       I-ORG       0.69      0.47      0.56       835
       I-PER       0.40      0.86      0.55      1156
           O       0.98      0.99      0.98     38323

    accuracy                           0.92     46435
   macro avg       0.71      0.65      0.66     46435
weighted avg       0.93      0.92      0.92     46435



## SVM classifier analysis



**[6 points] e) Train a model that uses the embeddings of these words as inputs. Test again on the same data as in 2d. Generate a classification report and compare the results with the classifier you built in 2d.**

## [Points: 10] Exercise 2 (NERC): feature inspection using the [Annotated Corpus for Named Entity Recognition](https://www.kaggle.com/abhinavwalia95/entity-annotated-corpus)
**[6 points] a. Perform the same steps as in the previous exercise. Make sure you end up for both the training part (*df_train*) and the test part (*df_test*) with:**
* the features representation using **DictVectorizer**
* the NERC labels in a list

Please note that this is the same setup as in the previous exercise:
* load both train and test using:
    * list of dictionaries for features
    * list of NERC labels
* combine train and test features in a list and represent them using one hot encoding
* train using the training features and NERC labels

In [ ]:
import pandas

In [ ]:
##### Adapt the path to point to your local copy of NERC_datasets
path = 'nerc_datasets/ner_v2.csv'
kaggle_dataset = pandas.read_csv(path, error_bad_lines=False)

In [ ]:
len(kaggle_dataset)

In [ ]:
df_train = kaggle_dataset[:100000]
df_test = kaggle_dataset[100000:120000]
print(len(df_train), len(df_test))

**[4 points] b. Train and evaluate the model and provide the classification report:**
* use the SVM to predict NERC labels on the test data
* evaluate the performance of the SVM on the test data

Analyze the performance per NERC label.

## End of this notebook